In [4]:
import os
import pandas as pd

In [5]:
shot_usage = pd.read_csv('./data/shot_usageNEW.csv', index_col=0)

In [6]:
distribution_df = pd.DataFrame(index=shot_usage.index, columns=['n_imgs_RIS1', 'n_imgs_RIS2'])

In [7]:
type(shot_usage.at[13182, 'used_for_ris1'])

numpy.bool_

In [8]:
for shot_number in shot_usage.index:
    for _, _, imgs in os.walk(f'./imgs/{shot_number}'):
        n_imgs_RIS1 = 0
        n_imgs_RIS2 = 0
        for img in imgs:
            if 'RIS1' in img and shot_usage.at[shot_number, 'used_for_ris1']:
                n_imgs_RIS1 += 1
            elif 'RIS2' in img and shot_usage.at[shot_number, 'used_for_ris2']:
                n_imgs_RIS2 += 1
        distribution_df.at[shot_number, 'n_imgs_RIS1'] = n_imgs_RIS1
        distribution_df.at[shot_number, 'n_imgs_RIS2'] = n_imgs_RIS2

In [9]:
distribution_df

,n_imgs_RIS1,n_imgs_RIS2
shot,,
13182,0,0
16532,2176,2176
16534,2181,2181
16769,2178,2178
16773,2180,2180
...,...,...
20143,0,0
20144,0,0
20145,0,0


In [10]:
non_zero_dist_df = distribution_df[~(distribution_df.any(axis=1) == 0)]

In [11]:
a = non_zero_dist_df.sort_values(by='n_imgs_RIS2', ascending=False)

In [12]:
non_zero_dist_df.sum()

n_imgs_RIS1    153440
n_imgs_RIS2    112291
dtype: object

In [13]:
# Set the directory path
directory = '/compass/Shared/Users/bogdanov/ml_tokamak/imgs'

# Initialize a counter for the number of files
file_count = 0

# Walk through all subfolders and count the files
for root, dirs, files in os.walk(directory):
    file_count += len(files)

# Print the total number of files
print(f'Total number of files: {file_count}')

Total number of files: 378675


In [14]:
root.split('/')[-1]

'19393'

### Mode distribution

In [46]:
from pathlib import Path
import confinement_mode_classifier as cmc
import numpy as np

ris_option = 'RIS2'

shot_usage = pd.read_csv(f'/compass/Shared/Users/bogdanov/ml_tokamak/data/shot_usageNEW.csv')
shot_for_ris = shot_usage[shot_usage['used_for_ris2'] if ris_option == 'RIS2' else shot_usage['used_for_ris1']]
shot_numbers = shot_for_ris['shot']
shots_for_testing = shot_for_ris[shot_for_ris['used_as'] == 'test']['shot']
shots_for_validation = shot_for_ris[shot_for_ris['used_as'] == 'val']['shot']
shots_for_training = shot_for_ris[shot_for_ris['used_as'] == 'train']['shot']

path = Path(os.getcwd())

In [47]:
shot_df, test_df, val_df, train_df = cmc.load_and_split_dataframes(path,shot_numbers, shots_for_training, shots_for_testing, 
                                                                    shots_for_validation, use_ELMS=True, ris_option=ris_option,
                                                                    exponential_elm_decay=False)

In [48]:

dist_df = pd.DataFrame({'train_df': train_df['mode'].value_counts().values, 'val_df': val_df['mode'].value_counts().values, 'test_df': test_df['mode'].value_counts().values}, 
                       index=['L-mode', 'H-mode', 'ELM'])


In [50]:
import numpy as np
np.vstack([train_df['mode'].value_counts().values, val_df['mode'].value_counts().values, test_df['mode'].value_counts().values]).transpose()

array([[52170, 19423, 19207],
       [ 7182,  2132,  2408],
       [ 3170,   750,   601]])

In [51]:
dist_df

,train_df,val_df,test_df
L-mode,52170,19423,19207
H-mode,7182,2132,2408
ELM,3170,750,601
